# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [1]:
!pip install pytest schedule requests-mock pytest-mock

In [2]:
# Библиотеки, которые могут вам понадобиться
# При необходимости расширяйте список
import time
from pathlib import Path
import os
import requests
import re
import schedule
import pytest
from bs4 import BeautifulSoup
from datetime import datetime

## Задание 1. Сбор данных об одной книге (20 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [3]:
def get_book_data(book_url: str) -> dict:
    """
    Parse book data from book_url

    Args:
        book_url: url to get a book

    Returns:
        book info dictionary

    """
    page = requests.get(book_url, timeout=60)
    page.raise_for_status()
    page.encoding = "utf-8"
    soup = BeautifulSoup(page.text, "html.parser")
    result = {}

    try:
        result["title"] = soup.find("h1").get_text()
    except AttributeError:
        result["title"] = "not found"
    try:
        result["price"] = soup.find("p", attrs={"class": "price_color"}).get_text()
    except AttributeError:
        result["price"] = "not found"
    try:
        result["available"] = re.search(
            r"\d+", soup.find("p", attrs={"class": "instock availability"}).get_text()
        ).group(0)
    except AttributeError:
        result["available"] = "not found"
    try:
        result["description"] = (
            soup.find("div", attrs={"id": "product_description"}).find_next_sibling("p")
        ).get_text()
    except AttributeError:
        result["description"] = "not found"
    try:
        product_information = {}
        for tr in soup.find("table").find_all("tr"):
            product_information[tr.find("th").get_text()] = tr.find("td").get_text()
        result["product_information"] = product_information
    except AttributeError:
        result["product_information"] = {}

    return result

In [4]:
# Используйте для самопроверки
book_url = "http://books.toscrape.com/catalogue/out-of-print-city-lights-spotlight-no-14_536/index.html"
get_book_data(book_url)

{'title': 'Out of Print: City Lights Spotlight No. 14',
 'price': '£53.64',
 'available': '8',
 'description': 'The third full-length collectionby Julien Poirier, Out of Print is a truly bicoastal volume, reflecting the poet\'s years in New York as well as his return to his Bay Area roots. Consider it a meetinghouse between late New York School and contemporary California surrealism, a series of quips intercepted from America\'s underground poetry telegraph, or an absurdist mirror hel The third full-length collection\xa0by Julien Poirier, Out of Print is a truly bicoastal volume, reflecting the poet\'s years in New York as well as his return to his Bay Area roots. Consider it a meetinghouse between late New York School and contemporary California surrealism, a series of quips intercepted from America\'s underground poetry telegraph, or an absurdist mirror held up to consumerist culture."Welcome Julien Poirier! What a distinct inspired voice. His work is abundant in surprise. His musica

In [5]:
# Book without description
book_url = "http://books.toscrape.com/catalogue/alice-in-wonderland-alices-adventures-in-wonderland-1_5/index.html"
get_book_data(book_url)

{'title': "Alice in Wonderland (Alice's Adventures in Wonderland #1)",
 'price': '£55.53',
 'available': '1',
 'description': 'not found',
 'product_information': {'UPC': 'cd2a2a70dd5d176d',
  'Product Type': 'Books',
  'Price (excl. tax)': '£55.53',
  'Price (incl. tax)': '£55.53',
  'Tax': '£0.00',
  'Availability': 'In stock (1 available)',
  'Number of reviews': '0'}}

## Задание 2. Сбор данных обо всех книгах (20 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [6]:
# Добавляю переменную ROOT_PATH, чтобы корректно определить, куда класть артефакты
ROOT_PATH = Path(os.getcwd()).parent
print(ROOT_PATH)

/Users/l.tereshchenkova/Documents/MIPT/Python/HW/books_scraper


In [7]:
def scrape_books(is_save: bool = True, filename: str = "books_data.txt") -> list:
    """
    Parse books list, put information about books to file

    Args:
        is_save: sign if the result should be saved
        filename: filename for the result saving

    Returns:
        list of books

    """
    print("Start scraping.")
    root = "http://books.toscrape.com/catalogue/"
    i = 0
    result = []

    while True:
        i += 1
        page_url = re.sub(r"{N}", str(i), root + "page-{N}.html")
        response = requests.get(page_url, timeout=60)
        if response.status_code != 200:
            break
        soup = BeautifulSoup(response.text, "html.parser")
        books_soup = soup.find("ol").find_all("li")
        for book in books_soup:
            book_link = book.find("a")
            if book_link.has_attr("href"):
                href = book_link["href"]
                try:
                    book_data = get_book_data(root + href)
                except AttributeError as e:
                    print(f"get_book_data({root + href})." f"ERROR: {e}")
                    continue
                result.append(book_data)

    if is_save:
        with open(f"{ROOT_PATH}/artifacts/{filename}", "w", encoding="utf-8") as f:
            f.write("".join(str(result)))
    print(
        "Scraping is finished. "
        f"Checked {i - 1} pages, archived {len(result)} books."
        f"The result saved to {f"/artifacts/{filename}"}"
    )
    return result

In [8]:
# Проверка работоспособности функции
res = scrape_books(
    is_save=True, filename="books_data_notebook.txt"
)  # Допишите ваши аргументы
print(type(res), len(res))  # и проверки

Start scraping.
Scraping is finished. Checked 50 pages, archived 1000 books.The result saved to /artifacts/books_data_notebook.txt
<class 'list'> 1000


## Задание 3. Настройка регулярной выгрузки (10 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [ ]:
# НАЧАЛО ВАШЕГО РЕШЕНИЯ
schedule.clear()


@schedule.repeat(schedule.every().day.until('2025-11-01 14:59:59').at("14:40"))
def job():
    filename = f"books_data_{str(datetime.now().date())}.txt"
    scrape_books(is_save=True, filename=filename)


print(schedule.get_jobs())

while True:
    schedule.run_pending()
    time.sleep(120)
# КОНЕЦ ВАШЕГО РЕШЕНИЯ

[Every 1 day at 14:40:00 do job() (last run: [never], next run: 2025-11-01 14:40:00)]
Start scraping.
Scraping is finished. Checked 50 pages, archived 1000 books.The result saved to /artifacts/books_data_2025-11-01.txt


## Задание 4. Написание автотестов (15 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [10]:
# Ячейка для демонстрации работоспособности
# Сам код напишите в отдельном скрипте
! pytest ../tests

============================= test session starts ==============================
platform darwin -- Python 3.14.0, pytest-8.4.2, pluggy-1.6.0
rootdir: /Users/l.tereshchenkova/Documents/MIPT/Python/HW/books_scraper
configfile: pyproject.toml
plugins: mock-3.15.1, anyio-4.11.0, requests-mock-1.12.1
collected 7 items                                                              

../tests/book_scraper/test_get_book_data.py .                            [ 14%]
../tests/book_scraper/test_scrape_books.py ......                        [100%]

============================== 7 passed in 0.22s ===============================


## Задание 5. Оформление проекта на GitHub и работа с Git (35 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds_2025.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```